# 可选实验：逻辑回归的梯度下降

## 目标
在本实验中，你将：
- 探索逻辑回归的梯度下降更新代码。
- 在一个熟悉的数据集上探索梯度下降

In [1]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
plt.rcParams['font.size'] = 8
import copy, math
from lab_utils_common import  dlc, plot_data, plt_tumor_data, sigmoid, compute_cost_logistic
from plt_quad_logistic import plt_quad_logistic, plt_prob

## 数据集
让我们从决策边界实验中所用的同一个双特征数据集开始。

In [2]:
X_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

和之前一样，我们将用一个辅助函数来绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，标签为 $y=0$ 的数据点显示为黑色圆圈。

In [3]:
fig,ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X_train, y_train, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 逻辑回归的梯度下降
<img align="right" src="./图片/C1_W3_Logistic_gradient_descent.png"     style=" width:400px; padding: 10px; " >

> **图中文字（中文）**：梯度下降 Repeat { … } 线性回归 / 逻辑回归 线性回归中 f_{w,b}(x) = w·x + b

回忆一下，梯度下降算法使用梯度计算：
$$\\begin{align*}
&\\text{repeat until convergence:} \\; \\lbrace \\\\
&  \\; \\; \\;w_j = w_j -  \\alpha \\frac{\\partial J(\\mathbf{w},b)}{\\partial w_j} \\tag{1}  \\; & \\text{for j := 0..n-1} \\\\ 
&  \\; \\; \\;  \\; \\;b = b -  \\alpha \\frac{\\partial J(\\mathbf{w},b)}{\\partial b} \\\\
&\\rbrace
\\end{align*}$$

其中每次迭代对所有 $j$ 的 $w_j$ 同时更新，且
$$\\begin{align*}
\\frac{\\partial J(\\mathbf{w},b)}{\\partial w_j}  &= \\frac{1}{m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - \\mathbf{y}^{(i)})x_{j}^{(i)} \\tag{2} \\\\
\\frac{\\partial J(\\mathbf{w},b)}{\\partial b}  &= \\frac{1}{m} \\sum\\limits_{i = 0}^{m-1} (f_{\\mathbf{w},b}(\\mathbf{x}^{(i)}) - \\mathbf{y}^{(i)}) \\tag{3} 
\\end{align*}$$

* m 是数据集中的训练样本数
* $f_{\\mathbf{w},b}(x^{(i)})$ 是模型的预测，而 $y^{(i)}$ 是目标值
* 对于逻辑回归模型
    $z = \\mathbf{w} \\cdot \\mathbf{x} + b$  
    $f_{\\mathbf{w},b}(x) = g(z)$  
    其中 $g(z)$ 是 sigmoid 函数：  
    $g(z) = \\frac{1}{1+e^{-z}}$  
    


### 梯度下降的实现
梯度下降算法的实现包含两个部分：
- 实现上面公式 (1) 的循环。即下面的 `gradient_descent`，通常在可选实验和实践实验中已提供给你。
- 当前梯度的计算，即上面的公式 (2)、(3)。即下面的 `compute_gradient_logistic`。本周的实践实验将要求你实现它。

#### 计算梯度，代码说明
为所有 $w_j$ 和 $b$ 实现上面的公式 (2)、(3)。
实现方法有很多，下面概述其中一种：
- 初始化用于累加 `dj_dw` 和 `dj_db` 的变量
- 对每个样本
    - 计算该样本的误差 $g(\\mathbf{w} \\cdot \\mathbf{x}^{(i)} + b) - \\mathbf{y}^{(i)}$
    - 对该样本中的每个输入值 $x_{j}^{(i)}$
        - 把误差乘以输入 $x_{j}^{(i)}$，加到 `dj_dw` 对应的元素上。（上面的公式 2）
    - 把误差加到 `dj_db`（上面的公式 3）

- 把 `dj_db` 和 `dj_dw` 除以样本总数（m）
- 注意，numpy 中的 $\\mathbf{x}^{(i)}$ 是 `X[i,:]` 或 `X[i]`，而 $x_{j}^{(i)}$ 是 `X[i,j]`

In [4]:
def compute_gradient_logistic(X, y, w, b): 
    """计算线性回归的梯度
 
    参数：
      X : (ndarray 形状 (m,n)) 变量，例如房屋面积
      y : (ndarray 形状 (m,))  实际值
      w : (ndarray 形状 (n,))  模型参数
      b : (scalar)             模型参数
    返回
      dj_dw: (ndarray 形状 (n,)) 代价对参数 w 的梯度。
      dj_db: (scalar)             代价对参数 b 的梯度。"""
    m,n = X.shape
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)
        err_i  = f_wb_i  - y[i]    
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m
    dj_db = dj_db/m
        
    return dj_db, dj_dw  #对 dj_db 取索引以返回标量值

用下面的单元格检查梯度函数的实现。

In [5]:
X_tmp = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_tmp = np.array([0, 0, 0, 1, 1, 1])
w = np.array([2.,3.])
b = 1.
dj_db, dj_dw = compute_gradient_logistic(X_tmp, y_tmp, w, b)
print(f"dj_db, non-vectorized version: {dj_db}" )
print(f"dj_dw, non-vectorized version: {dj_dw.tolist()}" )

dj_db, non-vectorized version: 0.49861806546328574
dj_dw, non-vectorized version: [0.498333393278696, 0.49883942983996693]


**预期输出**

``` 
dj_db, non-vectorized version: 0.49861806546328574
dj_dw, non-vectorized version: [[0.498333393278696], [0.49883942983996693]]
```


#### 梯度下降代码
实现上面公式 (1) 的代码如下。花点时间把程序中的函数与上面的公式对照一下。

In [7]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters): 
    """执行批量梯度下降
   
    参数：
      X (ndarray): 形状 (m,n)    样本矩阵
      y (ndarray): 形状 (m,)     每个样本的目标值
      w_in (ndarray): 形状 (n,)  模型参数的初始值
      b_in (scalar):              模型参数的初始值
      alpha (float):              学习率
      num_iters (int):            运行梯度下降的迭代次数
     
    返回：
      w (ndarray): 形状 (n,)     更新后的参数值
      b (scalar):                 更新后的参数值"""
    # 训练样本数
    m = len(X)
    
    # 用于存放代价 J 以及每次迭代的 w，主要供后续绘图使用
    J_history = []
    w = copy.deepcopy(w_in)  # 避免在函数内修改全局变量 w
    b = b_in
    
    for i in range(num_iters):
        # 计算梯度并更新参数
        dj_db, dj_dw = compute_gradient_logistic(X, y, w, b)   

        # 用 w、b、alpha 和梯度更新参数
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # 每次迭代保存代价 J
        if i<100000:      # 防止资源耗尽
            J_history.append( compute_cost_logistic(X, y, w, b) )

        # 每 10 次迭代打印一次代价；若迭代次数少于 10，则打印所有迭代
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}   ")
        
    return w, b, J_history, #返回最终的 w、b 以及用于绘图的 J 历史


让我们在数据集上运行梯度下降。

In [8]:
w_in  = np.zeros_like(X_train[0])
b_in  = 0.
alpha = 0.1
num_iters = 10000

w_out, b_out, _ = gradient_descent(X_train, y_train, w_in, b_in, alpha, num_iters) 
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")

Iteration    0: Cost 0.684610468560574   
Iteration 1000: Cost 0.1590977666870456   
Iteration 2000: Cost 0.08460064176930081   
Iteration 3000: Cost 0.05705327279402531   
Iteration 4000: Cost 0.042907594216820076   
Iteration 5000: Cost 0.034338477298845684   
Iteration 6000: Cost 0.028603798022120097   
Iteration 7000: Cost 0.024501569608793   
Iteration 8000: Cost 0.02142370332569295   
Iteration 9000: Cost 0.019030137124109114   

updated parameters: w:[5.28 5.08], b:-14.222409982019837


#### 让我们绘制梯度下降的结果：

In [10]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
# 绘制概率
plt_prob(ax, w_out, b_out)

# 绘制原始数据
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')   
ax.axis([0, 4, 0, 3.5])
plot_data(X_train,y_train,ax)

# 绘制决策边界
x0 = -b_out/w_out[1]
x1 = -b_out/w_out[0]
ax.plot([0,x0],[x1,0], c=dlc["dlblue"], lw=1)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在上图中：
 - 阴影反映概率 y=1（决策边界之前的结果）
 - 决策边界是概率 = 0.5 处的那条线
 

## 另一个数据集
让我们回到单变量数据集。只有两个参数 $w$、$b$ 时，可以用等高线图绘制代价函数，从而更好地了解梯度下降在做什么。

In [11]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

和之前一样，我们将用一个辅助函数来绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，标签为 $y=0$ 的数据点显示为黑色圆圈。

In [12]:
fig,ax = plt.subplots(1,1,figsize=(4,3))
plt_tumor_data(x_train, y_train, ax)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

在下面的图中，请尝试：
- 在右上方的等高线图内点击，改变 $w$ 和 $b$。
    - 变化可能需要一两秒
    - 注意左上图中代价的变化
    - 注意代价是由每个样本的损失累加而成的（竖直虚线）
- 点击橙色按钮运行梯度下降。
    - 注意代价稳定下降（等高线图和代价图用的是 log(cost)）
    - 在等高线图中点击会为新的运行重置模型

In [13]:
w_range = np.array([-1, 7])
b_range = np.array([1, -14])
quad = plt_quad_logistic( x_train, y_train, w_range, b_range )

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 恭喜！
你已经：
- 检查了逻辑回归中计算梯度的公式与实现
- 在以下场景中运用了这些程序
    - 探索单变量数据集
    - 探索双变量数据集